# 03 — Phase 2 eval + ablations (Amazon Reviews 2023, Electronics, balanced)
Runs text_only / plus_metadata / plus_image / plus_rag_reviews / plus_rag_specs / full_graph, plus a norm-vs-stdev dissonance ablation, on a per-rating-balanced slice.

In [ ]:
%cd /content
!rm -rf /content/repo
!git clone https://github.com/kvarun314/pes-mtech-project.git /content/repo
%cd /content/repo
%pip install -q -e ".[colab]"
import sys
sys.path.insert(0, "/content/repo/src")  # editable-install .pth is only read at
# interpreter startup, not picked up by an already-running kernel -- insert the
# path directly so the import below works without a runtime restart.
import agentic_sentiment
print("agentic_sentiment importable:", agentic_sentiment.__file__)

In [ ]:
import os
from huggingface_hub import login

# Colab Secrets panel (recommended): add a secret named HF_TOKEN, then
# grant this notebook access to it (toggle in the Secrets panel) — it
# reaches this cell via google.colab.userdata, NOT via os.environ.
# Accept the Llama-3 license at https://huggingface.co/meta-llama/Meta-Llama-3-8B
# before this token will work.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")  # raises if access wasn't granted — fix that, don't swallow it
except ImportError:
    hf_token = os.environ.get("HF_TOKEN", "")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN is empty. Set it in Colab Secrets (left sidebar) and grant this "
        "notebook access, or set os.environ['HF_TOKEN'] directly."
    )
login(hf_token)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/pes-mtech-project/results/run2"
import os; os.makedirs(RESULTS_DIR, exist_ok=True)

## Load Amazon Reviews 2023 — Electronics

In [ ]:
import random
from datasets import load_dataset

# Reviews are a single JSONL file; metadata is split into several parquet
# shards for a category this size (confirmed live: Electronics metadata is
# NOT single-shard like All_Beauty was in Run 1 -- a hardcoded "-of-00001"
# filename 404s). The hf:// glob lets datasets resolve however many shards
# actually exist instead of hardcoding a shard count that can drift.
_HF = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main"
_REVIEW_FILE = f"{_HF}/raw/review_categories/Electronics.jsonl"
_META_GLOB = "hf://datasets/McAuley-Lab/Amazon-Reviews-2023/raw_meta_Electronics/*.parquet"

print("Streaming reviews from:", _REVIEW_FILE)
review_stream = load_dataset("json", data_files=_REVIEW_FILE, split="train", streaming=True)
reviews_pool = []
random.seed(42)
for r in review_stream:
    reviews_pool.append(r)
    if len(reviews_pool) >= 60000:  # oversample the balanced-slice target (2000), then stop streaming
        break
random.shuffle(reviews_pool)
print(f"  Pulled {len(reviews_pool)} candidate reviews.")

print("Loading metadata from:", _META_GLOB)
needed_asins = {str(r.get("parent_asin") or "") for r in reviews_pool}
needed_asins.discard("")
meta_ds = load_dataset("parquet", data_files=_META_GLOB, split="train")
# Filter by column (batched) rather than a 1.6M-row Python for-loop.
meta_ds = meta_ds.filter(
    lambda batch: [a in needed_asins for a in batch["parent_asin"]], batched=True
)
meta_by_asin = {}
for row in meta_ds:
    asin = str(row.get("parent_asin") or "")
    if asin and asin not in meta_by_asin:
        meta_by_asin[asin] = row
print(f"  Metadata found for {len(meta_by_asin)}/{len(needed_asins)} ASINs.")

In [ ]:
from agentic_sentiment.data.amazon2023 import load_balanced_slice, build_spec_records

rows = load_balanced_slice(reviews_pool, meta_by_asin, n_per_rating=400, seed=42)
print(len(rows), "balanced eval rows")

eval_asins = {r["asin"] for r in rows}
spec_records = build_spec_records({a: m for a, m in meta_by_asin.items() if a in eval_asins})
print(len(spec_records), "spec records for", len(eval_asins), "products")

## BLIP captions (before loading LLaMA, to save VRAM)

In [ ]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
import requests
from PIL import Image

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("cuda")
for row in rows:
    row["image_caption"] = None
    if row.get("image_url"):
        try:
            img = Image.open(requests.get(row["image_url"], timeout=5, stream=True).raw).convert("RGB")
            inputs = processor(img, return_tensors="pt").to("cuda")
            row["image_caption"] = processor.decode(blip.generate(**inputs)[0], skip_special_tokens=True)
        except Exception:
            pass
del blip
torch.cuda.empty_cache()

## Build spec store (text embeddings)
Uses `all-MiniLM-L6-v2` (sentence-transformers) as a documented, locally-runnable
substitute for CLIP (mission spec §5.2) -- it embeds spec text only, not images,
which is sufficient since the Prover matches review claims against spec *text*.

In [ ]:
from sentence_transformers import SentenceTransformer
from agentic_sentiment.rag.spec_store import SpecStore

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embed_fn = lambda text: embedder.encode(text).tolist()

spec_store = SpecStore(db_path="/content/spec_lancedb", embed_fn=embed_fn)
spec_store.build(spec_records)

## Load Phase 1 LoRA adapter from Task 12's run_dir

In [ ]:
import os
import glob
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Durable across runtimes: read from Drive, written by colab/01_train_phase1.ipynb
# (this notebook never assumes a bare /content/... path from another notebook).
adapter_candidates = sorted(glob.glob("/content/drive/MyDrive/pes-mtech-project/runs/*/best_adapter"))
if not adapter_candidates:
    raise RuntimeError(
        "No best_adapter found under .../pes-mtech-project/runs/ on Drive -- "
        "run colab/01_train_phase1.ipynb to completion first."
    )
adapter_dir = adapter_candidates[-1]
print("Using adapter:", adapter_dir)
_run_dir = os.path.dirname(adapter_dir)
_metrics_path = os.path.join(_run_dir, "metrics.json")
_best_path = os.path.join(_run_dir, "BEST.md")
if not os.path.exists(_metrics_path):
    print("WARNING: no metrics.json in", _run_dir, "-- this run may not have finished "
          "(or trained on dummy data if the Kaggle download failed). Check before continuing.")
if os.path.exists(_best_path):
    print("BEST.md:", open(_best_path).read().strip())
print("Adapter last modified:",
      __import__("datetime").datetime.fromtimestamp(os.path.getmtime(adapter_dir)))
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.truncation_side = "left"  # preserve the trailing "Sentiment (1-5):" cue if truncated
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype="float16"
)  # nf4, not the fp4 default -- must match how the adapter was trained (lora.py)
base = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B", quantization_config=bnb_config, device_map="auto",
    torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(base, adapter_dir)
model.eval()

from transformers import GenerationConfig
_gen_config = GenerationConfig(do_sample=False, max_new_tokens=128)

def llm_fn(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    out = model.generate(**inputs, generation_config=_gen_config)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

## Run every ablation (each resumable via its own checkpoint)

In [ ]:
from agentic_sentiment.eval.run_eval import ABLATIONS, run_ablation, compute_metrics
from agentic_sentiment.eval.checkpoint import read_records
import json

# plus_rag_specs and full_graph share identical flags and both use the spec
# store -- they are the same run by construction ("full_graph" = all context
# + real grounding + self-correction). Run every ablation once; alias
# plus_rag_specs from full_graph's result instead of paying for it twice.
RUN_NAMES = [n for n in ABLATIONS if n != "plus_rag_specs"]

all_metrics = {}
for name in RUN_NAMES:
    store = spec_store if name == "full_graph" else None
    ckpt = f"{RESULTS_DIR}/{name}_checkpoint.jsonl"
    run_ablation(name, rows, llm_fn=llm_fn, checkpoint_path=ckpt, spec_store=store)
    records = read_records(ckpt)
    all_metrics[name] = compute_metrics(records)
    print(name, all_metrics[name])

all_metrics["plus_rag_specs"] = all_metrics["full_graph"]
print("plus_rag_specs (aliased from full_graph)", all_metrics["plus_rag_specs"])

with open(f"{RESULTS_DIR}/metrics_summary.json", "w") as f:
    json.dump(all_metrics, f, indent=2)

## Dissonance-formula ablation (norm vs. Run 1 stdev)

In [ ]:
from agentic_sentiment.eval.checkpoint import read_records

# full_graph above already used dissonance_method="norm" (the default) --
# reuse that run instead of re-executing an identical graph. Only "stdev"
# is a genuinely different configuration.
dissonance_results = {"norm": all_metrics["full_graph"]}
print("norm (reused from full_graph)", dissonance_results["norm"])

ckpt = f"{RESULTS_DIR}/dissonance_stdev_checkpoint.jsonl"
run_ablation("full_graph", rows, llm_fn=llm_fn, checkpoint_path=ckpt,
             spec_store=spec_store, dissonance_method="stdev")
records = read_records(ckpt)
dissonance_results["stdev"] = compute_metrics(records)
print("stdev", dissonance_results["stdev"])

with open(f"{RESULTS_DIR}/dissonance_ablation_metrics.json", "w") as f:
    json.dump(dissonance_results, f, indent=2)

In [ ]:
import shutil, os, fnmatch
from agentic_sentiment.eval.results_log import format_results_entry
from agentic_sentiment.colab_sync import push_repo_changes

# Copy only the files this notebook itself produces -- an explicit
# allowlist, not "whatever happens to be in this Drive folder" (that
# folder is user-writable and could pick up a stray file, or
# something too large for a normal git push, if anything else ever
# writes there).
_ALLOWED_RESULT_PATTERNS = ("*_checkpoint.jsonl", "metrics_summary.json", "dissonance_ablation_metrics.json")
repo_results_dir = "/content/repo/results/run2"
os.makedirs(repo_results_dir, exist_ok=True)
copied = []
for name in os.listdir(RESULTS_DIR):
    if any(fnmatch.fnmatch(name, pat) for pat in _ALLOWED_RESULT_PATTERNS):
        shutil.copy2(os.path.join(RESULTS_DIR, name), os.path.join(repo_results_dir, name))
        copied.append(name)
print(f"Copied {len(copied)} result file(s) into {repo_results_dir}:", copied)

entry = format_results_entry(
    run_label="Phase 2 ablations (Amazon Reviews 2023, Electronics, balanced)",
    context={
        "dataset": "Amazon Reviews 2023, Electronics", "n_per_rating": 400,
        "seed": 42, "n_eval_rows": len(rows), "dissonance_threshold": 0.4,
    },
    metrics_by_name={**all_metrics, **{f"dissonance_{k}": v for k, v in dissonance_results.items()}},
)

# Use a fine-grained PAT scoped to just this repo (Contents: read/write).
# A missing GITHUB_TOKEN is fine -- this step is optional documentation.
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = os.environ.get("GITHUB_TOKEN", "")

if not github_token:
    print("GITHUB_TOKEN not set -- skipping auto-push. RESULTS.md entry:")
    print(entry)
else:
    results_path = "/content/repo/docs/RESULTS.md"
    existing_results = open(results_path).read() if os.path.exists(results_path) else ""
    if entry not in existing_results:
        with open(results_path, "a") as f:
            f.write(entry)
    print(push_repo_changes(
        repo_dir="/content/repo",
        paths=["results/run2", "docs/RESULTS.md"],
        message="Phase 2 eval + ablations, Run 2 (Electronics, balanced)", token=github_token,
    ))